In [1]:
import os
import glob
from enum import Enum

from torchvision.ops import box_iou

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [2]:
STORAGE_DIR = "Z:\\starcraft-vision\\"

DATA_DIR = os.path.join(STORAGE_DIR, "data")
GROUND_TRUTH_DIR = os.path.join(DATA_DIR, "label", "dst")
PREDICTIONS_DIR = os.path.join(STORAGE_DIR, "predictions")

In [3]:
# REPLAYS = ['36.rep', '212.rep', '438.rep', '522.rep', '1660.rep', '6254.rep']
# REPLAYS = ['1559.rep', '1628.rep', '2351.rep', '6219.rep', '11251.rep']
REPLAYS = ['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']

In [4]:
TARGET_MODEL = "all_correct_win4_b16_20250826_045543/model_030"
# TARGET_MODEL = "test_input_model/model_020"

In [5]:
ground_truth_json_path = [
    f for name in REPLAYS
    for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, name, '*.json'))
]

In [6]:
ground_truth_json_path


['Z:\\starcraft-vision\\data\\label\\dst\\275.rep\\all_correct.json',
 'Z:\\starcraft-vision\\data\\label\\dst\\1725.rep\\all_correct.json',
 'Z:\\starcraft-vision\\data\\label\\dst\\3613.rep\\all_correct.json',
 'Z:\\starcraft-vision\\data\\label\\dst\\4520.rep\\all_correct.json',
 'Z:\\starcraft-vision\\data\\label\\dst\\4664.rep\\all_correct.json']

In [7]:
def read_json_files(directory, label_method='all_correct', replays=None):
    data = []

    if replays:
        targets = [os.path.join(rep, f"{label_method}.json") for rep in replays]
    else:
        targets = os.listdir(directory)

    for filename in targets:
        if filename.endswith(".json"):
            filepath = os.path.join(directory, filename)
            if os.path.isfile(filepath):
                with open(filepath, "r", encoding="utf-8") as f:
                    data.append(json.load(f))

    return data

In [8]:
ground_truth_data = read_json_files(GROUND_TRUTH_DIR, label_method='all_correct', replays=REPLAYS)

In [9]:
prediction_data = read_json_files(os.path.join(PREDICTIONS_DIR, TARGET_MODEL), label_method='all_correct', replays=REPLAYS)

In [10]:
def get_annotations(data, replay_suffix=".rep"):
    frames = []

    for item in (data or []):
        anns = item.get("annotations", [])
        if not anns:
            continue

        df = pd.DataFrame(anns)

        desc = (item.get("info") or {}).get("description", "") or ""
        replay_id = None
        if isinstance(desc, str) and desc.strip():
            replay_id = desc.strip().split()[-1]

        if replay_id is None:
            replay_id = "unknown"

        if replay_suffix and not str(replay_id).endswith(replay_suffix):
            replay_id = f"{replay_id}{replay_suffix}"

        df["replay"] = replay_id
        frames.append(df)

    base_cols = ["replay", "id", "image_id", "category_id", "bbox", "area", "segmentation", "iscrowd"]
    if not frames:
        return pd.DataFrame(columns=base_cols + ["score"])

    has_score = any("score" in f.columns for f in frames)

    wanted_cols = base_cols + (["score"] if has_score else [])
    for i, f in enumerate(frames):
        for col in wanted_cols:
            if col not in f.columns:
                frames[i][col] = pd.NA

    annotations = pd.concat(frames, ignore_index=True)

    return annotations[wanted_cols]

In [11]:
annotation_ground_truth = get_annotations(ground_truth_data)
annotation_prediction = get_annotations(prediction_data)

In [12]:
def dataframe_topk_score(dataframe, K=1):
    topk_df = (
      dataframe.sort_values('score', ascending=False)
        .groupby(['replay', 'image_id'], group_keys=False)
        .head(K)
    )
    return topk_df.sort_values(['replay', 'image_id'])

In [13]:
top1_annotation_prediction = dataframe_topk_score(annotation_prediction, K=1)

In [14]:
annotation_ground_truth["idx"] = annotation_ground_truth.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_ground_truth = annotation_ground_truth.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_ground_truth.columns = [f"gt_{i}" for i in wide_ground_truth.columns]
wide_ground_truth.reset_index(inplace=True)


In [15]:
top1_annotation_prediction["idx"] = top1_annotation_prediction.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction = top1_annotation_prediction.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction.columns = [f"pred_{i}" for i in wide_prediction.columns]
wide_prediction.reset_index(inplace=True)

In [16]:
result_inference = pd.merge(wide_ground_truth, wide_prediction,on=["replay", "image_id"], how="outer")
result_inference = result_inference.bfill()
result_inference = result_inference.dropna()

new_cols = []

for c in result_inference.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback

result_inference.columns = pd.MultiIndex.from_tuples(new_cols)

In [17]:
result_inference.head()

meta                         gt                                      \
     replay image_id                 0                 1                 2   
0  1725.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  1725.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  1725.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  1725.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  1725.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                     pred  
                  3                 4                   0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]

In [18]:
def add_gt_union_metrics(
    df: pd.DataFrame,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    grid_w = 128, grid_h = 128,
    orig_w = 3456, orig_h = 3722,
    kernel_x = 20, kernel_y = 12,
    use_bbox_size = False,
    thresholds = (0.0, 0.3, 0.5),
):
    def _xywh_to_rect(x, y, w, h):
        if not use_bbox_size:
            w, h = kernel_x, kernel_y
        x = int(np.round(x * 32 / orig_w * (grid_w - kernel_x)))
        y = int(np.round(y * 32 / orig_h * (grid_h - kernel_y)))
        return x, y, w, h

    out_cols = []
    for i in range(5):
        out_cols.append((f"gt_{i}", "ic_ratio"))
        out_cols.append((f"gt_{i}", "ic_multi"))
        for t in thresholds:
            out_cols.append((f"gt_{i}", f"ic@{int(round(t*100)):03d}"))
            
    for c in out_cols:
        if c not in df.columns:
            df[c] = np.nan

    for i in df.index:
        total_tiles = np.zeros((grid_h, grid_w), dtype=np.int16)

        for gt_col in gt_cols:
            bx = df.at[i, gt_col] if gt_col in df.columns else None
            if bx is None or (isinstance(bx, float) and np.isnan(bx)):
                continue
            x, y, w, h = (bx if len(bx) == 4 else tuple(bx)[:4])
            rx, ry, rw, rh = _xywh_to_rect(x, y, w, h)
            total_tiles[rx:rx+rw, ry:ry+rh] += 1
        
            pred_tiles = np.zeros((grid_h, grid_w), dtype=bool)
            have_pred = False
            for pred_col in pred_cols:
                if pred_col not in df.columns:
                    continue
                bx = df.at[i, pred_col]
                if bx is None or (isinstance(bx, float) and np.isnan(bx)):
                    continue
                x, y, w, h = (bx if len(bx) == 4 else tuple(bx)[:4])
                rx, ry, rw, rh = _xywh_to_rect(x, y, w, h)
                pred_tiles[rx:rx+rw, ry:ry+rh] = True
                have_pred = True
            
            if not have_pred or pred_tiles.sum() == 0:
                df.at[i, (f"{gt_col}", "ic_multi")] = np.nan
                df.at[i, (f"{gt_col}", "ic_ratio")] = 0.0
                for thr in thresholds:
                    df.at[i, (f"{gt_col}", f"ic@{int(round(thr*100)):03d}")] = False
                continue
            
            pred_tiles = total_tiles[pred_tiles]
            intersect_multi = pred_tiles.mean()
            pred_tiles = np.where(pred_tiles == 0, pred_tiles, 1)
            intersection = pred_tiles.mean()

            df.at[i, ("_".join(gt_col), "ic_multi")] = intersect_multi
            df.at[i, ("_".join(gt_col), "ic_ratio")] = intersection

            for thr in thresholds:
                df.at[i, ("_".join(gt_col), f"ic@{int(round(thr*100)):03d}")] = float(intersection > thr if thr == 0.0 else intersection >= thr)
            
    return df


In [19]:
result_inference = add_gt_union_metrics(
    result_inference,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

In [20]:
gt_cols = result_inference.columns[
    result_inference.columns.get_level_values(0).str.startswith("gt_")
]

g = result_inference.groupby(("meta", "replay"))[gt_cols].mean()

summary = g.T.groupby(level=1).mean().T

summary

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
1725.rep,0.702617,0.677568,0.643369,0.978558,0.566724
275.rep,0.616489,0.583229,0.540271,0.839811,0.473138
3613.rep,0.675795,0.654017,0.623606,0.944252,0.551309
4520.rep,0.630923,0.600514,0.561672,0.789380,0.491143
4664.rep,0.620442,0.593089,0.567200,0.880909,0.503681


In [21]:
summary.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.649253,0.621683,0.587224,0.886582,0.517199
std,0.038052,0.041572,0.043973,0.076572,0.040060
min,0.616489,0.583229,0.540271,0.789380,0.473138
25%,0.620442,0.593089,0.561672,0.839811,0.491143
50%,0.630923,0.600514,0.567200,0.880909,0.503681
75%,0.675795,0.654017,0.623606,0.944252,0.551309
max,0.702617,0.677568,0.643369,0.978558,0.566724
